# Задание 1

## Задание 1.1

### Подготовка и установка библиотек

In [ ]:
!pip install datasets

In [ ]:
!pip install nltk

In [ ]:
!pip install ipywidgets

In [1]:
import nltk # Natural Language Toolkit
import string
from datasets import load_dataset

In [ ]:
# Загрузка нужных данных NLTK:
# nltk.download('punkt') # Для токенизации
nltk.download('punkt_tab')
nltk.download('wordnet') # Для лемматизации
nltk.download('omw-1.4') # Часто нужно для корректной лемматизации (WordNet)

In [ ]:
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Загрузка стоп-слов
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [2]:
# Загружаем датасет emotion (Hugging Face Datasets)
dataset = load_dataset("emotion")

### Знакомство с датасетом

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [4]:
# Берём несколько примеров из обучающей выборки
example = []
text = []
n = 5
for i in range(n):
    example.append(dataset["train"][i])
    text.append(dataset["train"][i]["text"])

In [5]:
for i in range(n):
    print(example[i])

{'text': 'i didnt feel humiliated', 'label': 0}
{'text': 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'label': 0}
{'text': 'im grabbing a minute to post i feel greedy wrong', 'label': 3}
{'text': 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'label': 2}
{'text': 'i am feeling grouchy', 'label': 3}


In [6]:
print("Исходный текст:")
for i in range(n):
    print(text[i])

Исходный текст:
i didnt feel humiliated
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
im grabbing a minute to post i feel greedy wrong
i am ever feeling nostalgic about the fireplace i will know that it is still on the property
i am feeling grouchy


### Токенизация, стемминг, лемматизация

In [7]:
# Шаг 1: Токенизация без доп. очистки
print("\nТокены (без дополнительной очистки):")
tokens = []
for i in range(n):
    tokens.append(nltk.word_tokenize(text[i]))
    print(tokens[i])


Токены (без дополнительной очистки):
['i', 'didnt', 'feel', 'humiliated']
['i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake']
['im', 'grabbing', 'a', 'minute', 'to', 'post', 'i', 'feel', 'greedy', 'wrong']
['i', 'am', 'ever', 'feeling', 'nostalgic', 'about', 'the', 'fireplace', 'i', 'will', 'know', 'that', 'it', 'is', 'still', 'on', 'the', 'property']
['i', 'am', 'feeling', 'grouchy']


In [8]:
# Функция предобработки со стеммингом
def preprocess_with_stemming(text):
    # Приведение к нижнему регистру
    text = text.lower()
    # Удаление знаков препинания
    text = text.translate(str.maketrans('', '', string.punctuation)) # метод maketrans создаёт таблицу перевода (translation table)
                                                                     # str.maketrans(x, y, z) — обычно x заменяется на y, а символы из z удаляются
                                                                     # метод translate применяет эту таблицу к строке text
    # Токенизация
    tokens = nltk.word_tokenize(text)
    # Применение стемминга
    stemmer = nltk.PorterStemmer() #  - это класс, реализующий алгоритм стемминга
    stemmed_tokens = [stemmer.stem(token) for token in tokens] # stem() возвращает основу слова
    return stemmed_tokens

In [12]:
# Функция для преобразования тегов NLTK в теги WordNet
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# Создаём лемматизатор один раз (оптимизация)
lemmatizer = nltk.WordNetLemmatizer()

def preprocess_with_lemmatization(text):
    # Приведение к нижнему регистру
    text = text.lower()
    # Токенизация
    tokens = nltk.word_tokenize(text)
    # Определение частей речи
    tagged = nltk.pos_tag(tokens)
    # Лемматизация с учётом части речи и удаление пунктуации
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if token not in string.punctuation   # пунктуация удаляется ПОСЛЕ теггинга
    ]
    return lemmatized_tokens

### Сравнение результатов разных подходов

In [13]:
# Сравним результаты стемминга и лемматизации
stemmed_result = []
lemmatized_result = []
lemmatized_result_no_stop = []

for i in range(n):
    stemmed_result.append(preprocess_with_stemming(text[i]))

    lemm_result = preprocess_with_lemmatization(text[i])
    lemmatized_result.append(lemm_result)

    # Удаление стоп-слов из лемматизированного списка
    lemm_result_no_stop = [token for token in lemm_result if token not in stop_words]
    lemmatized_result_no_stop.append(lemm_result_no_stop)

for i in range(n):
    print(f"\n Пример №{i}")
    print("\nТокены после стемминга:")
    print(stemmed_result[i])
    print("\nТокены после лемматизации (с учётом POS):")
    print(lemmatized_result[i])
    print("\nТокены после лемматизации и удаления стоп-слов:")
    print(lemmatized_result_no_stop[i])
    print("\nТокены с примитивной предобработкой (только токенизация):")
    print(tokens[i])


 Пример №0

Токены после стемминга:
['i', 'didnt', 'feel', 'humili']

Токены после лемматизации (с учётом POS):
['i', 'didnt', 'feel', 'humiliate']

Токены после лемматизации и удаления стоп-слов:
['didnt', 'feel', 'humiliate']

Токены с примитивной предобработкой (только токенизация):
['i', 'didnt', 'feel', 'humiliated']

 Пример №1

Токены после стемминга:
['i', 'can', 'go', 'from', 'feel', 'so', 'hopeless', 'to', 'so', 'damn', 'hope', 'just', 'from', 'be', 'around', 'someon', 'who', 'care', 'and', 'is', 'awak']

Токены после лемматизации (с учётом POS):
['i', 'can', 'go', 'from', 'feel', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'be', 'around', 'someone', 'who', 'care', 'and', 'be', 'awake']

Токены после лемматизации и удаления стоп-слов:
['go', 'feel', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake']

Токены с примитивной предобработкой (только токенизация):
['i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned'

### Вывод

Стемминг грубо примерно обрезает слова до "основы".
Лемматизация - более "умный" алгоритм, ищущий "лемму", но для корректной работы, требующий определения части речи.

## Задание 1.2

Примеры отличий стемминга и лемматизации:

In [29]:
print("\nТокены после стемминга:")
print(stemmed_result[0])

print("\nТокены после лемматизации:")
print(lemmatized_result[0])

print("\nТокены с примитивной предобработкой:")
print(tokens[0])


Токены после стемминга:
['i', 'didnt', 'feel', 'humili']

Токены после лемматизации:
['i', 'didnt', 'feel', 'humiliated']

Токены с примитивной предобработкой:
['i', 'didnt', 'feel', 'humiliated']


Видим:

стемминг обрезает слово;

лемматизация, принимая все слова за существительные оставляет их в той же форме;

In [30]:
print("\nТокены после стемминга:")
print(stemmed_result[1])

print("\nТокены после лемматизации:")
print(lemmatized_result[1])

print("\nТокены с примитивной предобработкой:")
print(tokens[1])


Токены после стемминга:
['i', 'can', 'go', 'from', 'feel', 'so', 'hopeless', 'to', 'so', 'damn', 'hope', 'just', 'from', 'be', 'around', 'someon', 'who', 'care', 'and', 'is', 'awak']

Токены после лемматизации:
['i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'care', 'and', 'is', 'awake']

Токены с примитивной предобработкой:
['i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake']


Та же ситуация, НО:

лемматизация привела cares -> care(в единственное число);

стемминг получил такой же результат, но просто обрезав слово;


## Задание 1.3

Есть высказывание, которое нужно проверить экспериментально, выводя результат на экран:
**По умолчанию nltk.word_tokenize не удаляет знаки препинания, а просто выделяет их в
отдельные токены.** Например, если у вас есть фраза "Hello, world!", то результатом будет
что-то вроде ["Hello", ",", "world", "!"].
Если вы хотите **полностью исключить** пунктуацию из своего набора токенов (часто это нужно
в задачах классификации, чтобы «чистый» текст без знаков препинания был входом для модели),
то: **либо удаляете их до токенизации, либо удаляете их после токенизации**, отфильтровывая
токены, которые состоят только из знаков препинания (или содержат их).

### Без чистки

In [53]:
text_for_proof = "Hello, world! Hello-Hello!"
print(text_for_proof)

Hello, world! Hello-Hello!


In [54]:
tokens_for_proof_1 = nltk.word_tokenize(text_for_proof)
print(tokens_for_proof_1)

['Hello', ',', 'world', '!', 'Hello-Hello', '!']


### Чистка перед токенизацией

In [55]:
# Удаление знаков препинания
text_for_proof_clear_1 = text_for_proof.translate(str.maketrans('', '', string.punctuation))

tokens_for_proof_2 = nltk.word_tokenize(text_for_proof_clear_1)
print(tokens_for_proof_2)

['Hello', 'world', 'HelloHello']


### Чистка после токенизации

In [61]:
tokens_for_proof_3 = nltk.word_tokenize(text_for_proof)

print(tokens_for_proof_3)

# Оставляем только те токены, которые не состоят целиком из знаков препинания
cleaned_tokens_1 = [token for token in tokens_for_proof_3 if token not in string.punctuation]

# Для каждого токена удаляем все символы пунктуации
cleaned_tokens_2 = [token.translate(str.maketrans('', '', string.punctuation)) for token in tokens_for_proof_3]
# После удаления могут остаться пустые строки
cleaned_tokens_2 = [token for token in cleaned_tokens_2 if token]


print(cleaned_tokens_1)
print(cleaned_tokens_2)

['Hello', ',', 'world', '!', 'Hello-Hello', '!']
['Hello', 'world', 'Hello-Hello']
['Hello', 'world', 'HelloHello']


## Задание 1.4

### Знакомство с датасетом

In [15]:
from datasets import load_dataset
dataset = load_dataset("imdb")

In [17]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [18]:
# Берём несколько примеров из обучающей выборки
example = []
text = []
n = 5
for i in range(n):
    example.append(dataset["train"][i])
    text.append(dataset["train"][i]["text"])

In [19]:
for i in range(n):
    print(example[i])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [20]:
print("Исходный текст:")
for i in range(n):
    print(text[i])

Исходный текст:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and 

### Токенизация, стемминг, лемматизация

In [21]:
# Шаг 1: Токенизация без доп. очистки
print("\nТокены (без дополнительной очистки):")
tokens = []
for i in range(n):
    tokens.append(nltk.word_tokenize(text[i]))
    print(tokens[i])


Токены (без дополнительной очистки):
['I', 'rented', 'I', 'AM', 'CURIOUS-YELLOW', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967', '.', 'I', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized', 'by', 'U.S.', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', ',', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', '``', 'controversial', "''", 'I', 'really', 'had', 'to', 'see', 'this', 'for', 'myself.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'The', 'plot', 'is', 'centered', 'around', 'a', 'young', 'Swedish', 'drama', 'student', 'named', 'Lena', 'who', 'wants', 'to', 'learn', 'everything', 'she', 'can', 'about', 'life', '.', 'In', 'particular', 'she', 'wants', 'to', 'focus', 'her', 'attentions', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average', 'Swede', 'thought', 'about', 'certain', 'political'

### Сравнение результатов разных подходов

In [23]:
# Сравним результаты стемминга и лемматизации
stemmed_result = []
lemmatized_result = []
lemmatized_result_no_stop = []

for i in range(n):
    stemmed_result.append(preprocess_with_stemming(text[i]))

    lemm_result = preprocess_with_lemmatization(text[i])
    lemmatized_result.append(lemm_result)

    lemm_result_no_stop = [token for token in lemm_result if token not in stop_words]
    lemmatized_result_no_stop.append(lemm_result_no_stop)

for i in range(n):
    print(f"\n Пример №{i}")
    print("\nТокены после стемминга:")
    print(stemmed_result[i])
    print("\nТокены после лемматизации (с учётом POS):")
    print(lemmatized_result[i])
    print("\nТокены после лемматизации и удаления стоп-слов:")
    print(lemmatized_result_no_stop[i])
    print("\nТокены с примитивной предобработкой (только токенизация):")
    print(tokens[i])


 Пример №0

Токены после стемминга:
['i', 'rent', 'i', 'am', 'curiousyellow', 'from', 'my', 'video', 'store', 'becaus', 'of', 'all', 'the', 'controversi', 'that', 'surround', 'it', 'when', 'it', 'wa', 'first', 'releas', 'in', '1967', 'i', 'also', 'heard', 'that', 'at', 'first', 'it', 'wa', 'seiz', 'by', 'us', 'custom', 'if', 'it', 'ever', 'tri', 'to', 'enter', 'thi', 'countri', 'therefor', 'be', 'a', 'fan', 'of', 'film', 'consid', 'controversi', 'i', 'realli', 'had', 'to', 'see', 'thi', 'for', 'myselfbr', 'br', 'the', 'plot', 'is', 'center', 'around', 'a', 'young', 'swedish', 'drama', 'student', 'name', 'lena', 'who', 'want', 'to', 'learn', 'everyth', 'she', 'can', 'about', 'life', 'in', 'particular', 'she', 'want', 'to', 'focu', 'her', 'attent', 'to', 'make', 'some', 'sort', 'of', 'documentari', 'on', 'what', 'the', 'averag', 'swede', 'thought', 'about', 'certain', 'polit', 'issu', 'such', 'as', 'the', 'vietnam', 'war', 'and', 'race', 'issu', 'in', 'the', 'unit', 'state', 'in', 'betw

## Задание 1.5 Векторное представление

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

raw_docs = text[:5]
lemmatized_docs = [preprocess_with_lemmatization(doc) for doc in raw_docs]

Вариант 1 – склеиваем леммы в строки и подаём в CountVectorizer

In [25]:
# Вариант 1: лемматизированные документы в виде строк
docs_as_strings = [' '.join(tokens) for tokens in lemmatized_docs]
vectorizer1 = CountVectorizer(binary=True)
X1 = vectorizer1.fit_transform(docs_as_strings)

print("Вариант 1 (строки из лемм):")
print("Размер словаря:", len(vectorizer1.vocabulary_))
print("Матрица X (первые 5 документов):\n", X1.toarray())
print("Словарь (первые 10):", list(vectorizer1.vocabulary_.keys())[:10])

Вариант 1 (строки из лемм):
Размер словаря: 458
Матрица X (первые 5 документов):
 [[0 0 1 ... 1 0 1]
 [0 1 0 ... 1 1 0]
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 1 0]
 [0 0 0 ... 0 1 1]]
Словарь (первые 10): ['rent', 'be', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of']


Вариант 2 – передаём сырые строки и кастомный токенизатор

In [26]:
# Вариант 2: кастомный токенизатор (функция лемматизации)
# Важно: в функции уже есть lower(), поэтому lowercase=False
vectorizer2 = CountVectorizer(binary=True,
                              tokenizer=preprocess_with_lemmatization,
                              lowercase=False,
                              token_pattern=None)
X2 = vectorizer2.fit_transform(raw_docs)

print("\nВариант 2 (кастомный токенизатор):")
print("Размер словаря:", len(vectorizer2.vocabulary_))
print("Матрица X:\n", X2.toarray())
print("Словарь (первые 10):", list(vectorizer2.vocabulary_.keys())[:10])


Вариант 2 (кастомный токенизатор):
Размер словаря: 473
Матрица X:
 [[1 0 1 ... 0 0 1]
 [1 1 1 ... 1 1 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 1 0]
 [1 0 0 ... 0 1 1]]
Словарь (первые 10): ['i', 'rent', 'be', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of']


Вариант 3 – сырые строки без лемматизации (стандартный CountVectorizer)

In [27]:
# Вариант 3: стандартный CountVectorizer (без лемматизации)
vectorizer3 = CountVectorizer(binary=True)
X3 = vectorizer3.fit_transform(raw_docs)

print("\nВариант 3 (без лемматизации):")
print("Размер словаря:", len(vectorizer3.vocabulary_))
print("Матрица X:\n", X3.toarray())
print("Словарь (первые 10):", list(vectorizer3.vocabulary_.keys())[:10])


Вариант 3 (без лемматизации):
Размер словаря: 497
Матрица X:
 [[0 0 1 ... 1 0 1]
 [0 1 0 ... 1 1 0]
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 1 0]
 [0 0 0 ... 0 1 1]]
Словарь (первые 10): ['rented', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of']


Сравнение размеров словарей

In [28]:
print(f"Размер словаря (Вариант 1): {X1.shape[1]}")
print(f"Размер словаря (Вариант 2): {X2.shape[1]}")
print(f"Размер словаря (Вариант 3): {X3.shape[1]}")

Размер словаря (Вариант 1): 458
Размер словаря (Вариант 2): 473
Размер словаря (Вариант 3): 497
